# Explainable Spatio-Temporal Traffic Congestion Prediction
## Using CNN-LSTM with SMOTE and SHAP-based Explainability
**Dataset:** Metro Interstate Traffic Volume  
**Model:** CNN-LSTM  
**Techniques:** SMOTE (class balancing), SHAP (explainability)

## Cell 1 — Install Libraries

In [ ]:
!pip install imbalanced-learn shap --quiet
print("✅ Libraries installed")

## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

import shap
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("✅ All imports successful")

## Cell 3 — Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload Metro_Interstate_Traffic_Volume.csv

In [ ]:
df = pd.read_csv('Metro_Interstate_Traffic_Volume.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nFirst 5 rows:")
df.head()

## Cell 4 — EDA (Exploratory Data Analysis)

In [ ]:
# Parse datetime and extract features
df['date_time'] = pd.to_datetime(df['date_time'], dayfirst=True)
df['hour']        = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
df['month']       = df['date_time'].dt.month
df['is_weekend']  = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['temp']        = df['temp'] - 273.15  # Kelvin to Celsius
df['holiday']     = df['holiday'].fillna('None')
df['is_holiday']  = df['holiday'].apply(lambda x: 0 if x == 'None' else 1)
df['rain_1h']     = df['rain_1h'].clip(upper=100)  # Cap outliers

# Create congestion labels
def label_congestion(vol):
    if vol <= 1000:
        return 'Low'
    elif vol <= 3500:
        return 'Medium'
    else:
        return 'High'

df['congestion_level'] = df['traffic_volume'].apply(label_congestion)

# Plot 1 — Traffic Volume Distribution
plt.figure(figsize=(10, 4))
plt.hist(df['traffic_volume'], bins=50, color='steelblue', edgecolor='black')
plt.title('Traffic Volume Distribution')
plt.xlabel('Traffic Volume')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Plot 2 — Average Traffic by Hour
plt.figure(figsize=(10, 4))
df.groupby('hour')['traffic_volume'].mean().plot(kind='bar', color='coral')
plt.title('Average Traffic Volume by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Avg Traffic Volume')
plt.tight_layout()
plt.show()

# Plot 3 — Average Traffic by Day of Week
plt.figure(figsize=(10, 4))
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df.groupby('day_of_week')['traffic_volume'].mean().plot(kind='bar', color='mediumseagreen')
plt.xticks(range(7), days)
plt.title('Average Traffic Volume by Day of Week')
plt.xlabel('Day')
plt.ylabel('Avg Traffic Volume')
plt.tight_layout()
plt.show()

# Plot 4 — Congestion Class Distribution
plt.figure(figsize=(6, 4))
df['congestion_level'].value_counts().plot(kind='bar', color=['red', 'orange', 'green'])
plt.title('Congestion Level Distribution (Before SMOTE)')
plt.xlabel('Congestion Level')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print("Congestion distribution:")
print(df['congestion_level'].value_counts())
print("\nPercentage:")
print(df['congestion_level'].value_counts(normalize=True) * 100)

## Cell 5 — Preprocessing + SMOTE

In [ ]:
# Sort by time
df = df.sort_values('date_time').reset_index(drop=True)

# Encode weather
le_weather = LabelEncoder()
df['weather_encoded'] = le_weather.fit_transform(df['weather_main'])

# Encode congestion label
le_congestion = LabelEncoder()
df['congestion_encoded'] = le_congestion.fit_transform(df['congestion_level'])
print("Label mapping:", dict(zip(le_congestion.classes_,
      le_congestion.transform(le_congestion.classes_))))

# Select features
FEATURES = [
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday',
    'temp', 'rain_1h', 'snow_1h', 'clouds_all', 'weather_encoded'
]
TARGET = 'congestion_encoded'

X = df[FEATURES].values
y = df[TARGET].values

# Normalize
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Create sequences FIRST (before SMOTE)
SEQUENCE_LENGTH = 24

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y, SEQUENCE_LENGTH)
print(f"Sequence shape: {X_seq.shape}")
print(f"Label distribution before SMOTE:")
print(pd.Series(y_seq).value_counts())

# Train test split BEFORE SMOTE
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42, shuffle=False
)

# Apply SMOTE only on training data
n_samples, seq_len, n_features = X_train.shape
X_train_2d = X_train.reshape(n_samples, seq_len * n_features)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_2d, y_train)

# Reshape back to 3D
X_train_resampled = X_train_resampled.reshape(-1, seq_len, n_features)

print(f"\nAfter SMOTE training distribution:")
print(pd.Series(y_train_resampled).value_counts())
print(f"\nTraining set: {X_train_resampled.shape}")
print(f"Test set:     {X_test.shape}")
print("\n✅ Preprocessing complete!")

## Cell 6 — Build and Train CNN-LSTM Model

In [ ]:
num_classes = 3
y_train_cat = to_categorical(y_train_resampled, num_classes)
y_test_cat  = to_categorical(y_test, num_classes)

# Build model
model = Sequential([
    # CNN Block — extracts spatial/local patterns
    Conv1D(filters=64, kernel_size=3, activation='relu',
           input_shape=(SEQUENCE_LENGTH, len(FEATURES))),
    BatchNormalization(),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    # LSTM Block — learns temporal dependencies
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64, return_sequences=False),
    Dropout(0.3),

    # Output Block
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=3, verbose=1
)

# Train
history = model.fit(
    X_train_resampled, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\n✅ Training complete!")

## Cell 7 — Training Curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## Cell 8 — Evaluation

In [ ]:
# Predictions
y_pred_prob = model.predict(X_test)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = y_test

# Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred,
      target_names=le_congestion.classes_))

# Overall Accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {acc*100:.2f}%")

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_congestion.classes_,
            yticklabels=le_congestion.classes_)
plt.title('Confusion Matrix — CNN-LSTM')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Cell 9 — SHAP Explainability

In [ ]:
import shap

# Use a small background sample for SHAP
background = X_train_resampled[:200]
test_sample = X_test[:100]

# DeepSHAP explainer
explainer = shap.DeepExplainer(model, background)
shap_values = explainer.shap_values(test_sample)

# Reshape for summary plot
# shap_values shape: (n_classes, n_samples, seq_len, n_features)
# Average across sequence length
shap_mean = np.mean(np.abs(shap_values[0]), axis=1)  # High class

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_mean,
    features=test_sample.mean(axis=1),
    feature_names=FEATURES,
    plot_type='bar',
    show=True
)
plt.title('SHAP Feature Importance — High Congestion Class')
plt.tight_layout()
plt.show()

print("\nFeature importance ranking (High Congestion):")
importance = pd.DataFrame({
    'Feature': FEATURES,
    'SHAP Value': shap_mean.mean(axis=0)
}).sort_values('SHAP Value', ascending=False)
print(importance)